# 11. 优化算法（章节总览）

> 笔记来源：《动手学深度学习（PyTorch 版）》第 11 章目录页：
> [11. 优化算法](https://zh-v2.d2l.ai/chapter_optimization/index.html)
> 本笔记是该章的**导读与总览**：梳理全章主线、核心公式与算法演化脉络；各小节细节可再深入对应小节。

到目前为止，本书已经用过许多优化算法训练深度学习模型：优化算法负责持续更新模型参数、最小化损失函数。任何把优化当"黑盒咒语"（SGD、Adam……）的人都能用起来，但要用好，必须深入原理：

- **效率**：训练复杂模型可能耗时数小时、数天甚至数周，优化算法直接决定训练效率；
- **调参**：理解各算法的原则与超参数的作用，才能有针对性地调参、提升模型性能；
- **非凸**：深度学习中的优化问题几乎都是**非凸**的，但在**凸问题**背景下设计和分析算法非常有启发性——因此本章先讲凸性入门，再在凸目标上证明简单 SGD 的收敛性质。

全章主线一句话：

**GD →（去计算瓶颈）SGD →（平滑噪声与病态条件）动量法 →（逐维自适应学习率）AdaGrad →（修"衰减到零"缺陷）RMSProp →（去学习率量纲）AdaDelta →（集大成）Adam →（叠加时间调度）LR 调度器**

## 全章路线图

| 小节 | 主题 | 核心思想 | PyTorch API |
| --- | --- | --- | --- |
| 11.1 | 优化和深度学习 | 优化只管训练误差；局部极小、鞍点、梯度消失是三大挑战 | — |
| 11.2 | 凸性 | 凸函数下局部极小 = 全局极小，是理论分析的基石 | — |
| 11.3 | 梯度下降 | 沿负梯度方向迭代：$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \nabla f(\mathbf{w}_t)$ | — |
| 11.4 | 随机梯度下降 | 用单样本的**无偏**梯度估计替代全量梯度 | `torch.optim.SGD` |
| 11.5 | 小批量 SGD | 批量 $b$ 权衡计算效率与梯度方差 | `DataLoader` + `SGD` |
| 11.6 | 动量法 | 用指数加权平均的历史梯度平滑更新方向 | `SGD(momentum=β)` |
| 11.7 | AdaGrad | 按梯度平方**累加**逐维缩放学习率 | `torch.optim.Adagrad` |
| 11.8 | RMSProp | 用 **EMA** 替代累加，修复 AdaGrad 学习率衰减过快 | `torch.optim.RMSprop` |
| 11.9 | Adadelta | 更新量单位与参数一致，无需默认学习率 | `torch.optim.Adadelta` |
| 11.10 | Adam | 动量 + RMSProp + 偏差修正，实践默认首选 | `torch.optim.Adam` |
| 11.11 | 学习率调度器 | 学习率随时间衰减 / 余弦退火 / 预热 | `torch.optim.lr_scheduler` |

## 11.1 优化和深度学习

**优化的目标**。深度学习中的"优化"通常指**最小化训练误差**（经验风险）：

$$
f(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^{n} l\left(\mathbf{x}_i, y_i, \mathbf{w}\right)
$$

而机器学习的真正目标是**减小泛化误差**。两者并不一致：把训练误差压得过低（优化做得"太好"）反而可能过拟合。所以实践中要配合更多数据、正则化等手段来控制泛化。

**深度学习中的优化挑战**（"非凸 + 高维 + 随机"为什么难）：

1. **局部最小值（Local Minima）**：非凸目标可能有很多局部极小；梯度为 0 且不是全局最优时会"卡住"。缓解：小批量噪声带来的随机性有时能帮助跳出局部极小。
2. **鞍点（Saddle Points）**：梯度为 0 但 Hessian 有正有负特征值的点——既非极大也非极小。高维参数空间中**鞍点远比局部极小常见**（所有方向曲率都为正的概率随维度指数下降）。
3. **梯度消失（Vanishing Gradients）**：如 sigmoid / tanh 在饱和区导数趋近 0，深层网络中梯度逐层衰减，优化近乎停滞。缓解：ReLU 等激活函数、残差连接、合理初始化。

（此外还有两大问题：**病态条件**（ill-conditioning，各方向曲率差异大）与**梯度噪声**，分别由 11.6 动量法与 11.4 / 11.5 的随机性分析来应对。）

## 11.2 凸性

**凸集**：集合 $\mathcal{X}$ 中任意两点的连线仍在集合内：

$$
\lambda x + (1-\lambda) y \in \mathcal{X}, \quad \forall x, y \in \mathcal{X},\ \lambda \in [0, 1]
$$

**凸函数**：定义域是凸集，且任意"弦"都在函数图像上方：

$$
\lambda f(x) + (1-\lambda) f(y) \ge f\left(\lambda x + (1-\lambda) y\right), \quad \forall x, y,\ \lambda \in [0, 1]
$$

二阶可导时的判据：Hessian 矩阵半正定，$\nabla^2 f \succeq 0$（如二次函数 $\frac{1}{2}\mathbf{w}^\top \mathbf{P} \mathbf{w} + \mathbf{q}^\top \mathbf{w} + r$ 凸 $\iff \mathbf{P} \succeq 0$）。

**Jensen 不等式**（凸性推论）：$E[f(x)] \ge f(E[x])$——凸函数的期望不小于期望处的函数值，这是许多收敛性证明的基本工具。

**为什么凸性重要**：

- 凸函数下，**局部极小值就是全局极小值**（反证法易证）——算法掉进任何局部极小都令人满意；
- 凸集约束下的优化可引入**拉格朗日函数** $L(\mathbf{x}, \alpha) = f(\mathbf{x}) + \sum_i \alpha_i c_i(\mathbf{x})$ 与对偶问题，是理解约束优化、SVM 等的基础。

深度学习的损失面一般非凸，但凸分析给出了算法行为的"理想参照系"——后续小节的理论收敛性证明都建立在凸（甚至强凸）假设上。

## 11.3 梯度下降

**一维**。泰勒展开 $f(x + \epsilon) = f(x) + \epsilon f'(x) + O(\epsilon^2)$ 说明：沿负梯度方向 $-\eta f'(x)$ 移动能让函数值下降，得到迭代：

$$
x_{t+1} = x_t - \eta \frac{\partial f}{\partial x}(x_t)
$$

**多元**。把导数换成梯度向量，同理：

$$
\mathbf{x}_{t+1} = \mathbf{x}_t - \eta \nabla f(\mathbf{x}_t)
$$

- $\eta$（学习率）太大 → 发散 / 震荡；太小 → 收敛极慢；
- 固定 $\eta$ 时，梯度下降在**病态条件问题**（各方向曲率差异大，如 $f = 0.1 w_1^2 + 2 w_2^2$）上会在大曲率方向震荡、小曲率方向进展缓慢——这正是 11.6 动量法的动机；
- **自适应方法**：根据已有观察预热学习率、或按维度缩放步长（预告 11.7–11.10）。

## 11.4 随机梯度下降

全量梯度的代价 $O(n)$ 太贵。SGD 每步随机均匀采样一个样本 $\xi \in \{1, \dots, n\}$，用**单样本损失**的梯度更新：

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \eta_t \nabla_{\mathbf{w}}\, l(\mathbf{w}_t, \xi_t)
$$

关键性质——**无偏**：随机梯度的期望恰好等于全量梯度：

$$
E_{\xi}\left[\nabla_{\mathbf{w}}\, l(\mathbf{w}, \xi)\right] = \frac{1}{n} \sum_{i=1}^{n} \nabla_{\mathbf{w}}\, l(\mathbf{w}, \mathbf{x}_i) = \nabla f(\mathbf{w})
$$

**动态学习率**：单样本梯度的噪声方差不随迭代衰减，因此需要让 $\eta_t$ 随时间衰减，常见形式：

$$
\eta_t = \eta_0 \cdot t^{-\alpha} \quad (\text{多项式衰减})， \qquad \eta_t = \eta_0 \cdot \beta^t \quad (\text{指数衰减})， \qquad \text{分段常数}
$$

**收敛性（凸目标）**：对凸函数、方差有界的随机梯度，SGD 收敛率为 $O(1/\sqrt{t})$（凸）/ $O(1/t)$（强凸），慢于全量 GD 的线性收敛——但每步代价只有 $1/n$，总体反而更划算。

**随机梯度与有限样本**：训练集本身也是随机样本，SGD 天然适合"边来数据边更新"的在线学习场景。

**Q：为什么需要随机梯度下降（SGD）？全量梯度下降（GD）有什么痛点？**

**A：** 在全量梯度下降中，每更新一次参数需要计算所有 $n$ 个样本的梯度并求平均，其计算代价为 $O(n)$。当训练数据量极大时，这种全量梯度的代价太贵，导致计算效率极低。

**Q：SGD 的核心原理与参数更新公式是什么？**
**A：** SGD 放弃了精确计算全量梯度，改为每步迭代均从 $\{1, \dots, n\}$ 中随机均匀采样一个样本 $\xi$。算法仅仅使用这**单样本损失**的梯度来进行参数更新，更新公式为：


$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta_t \nabla_{\mathbf{w}} l(\mathbf{w}_t, \xi_t)$$


这一改变使单步更新代价只有全量梯度的 $1/n$，总体计算反而更划算。此外，由于训练集本身也是随机样本，SGD 这种基于单一数据点的更新机制天然适合“边来数据边更新”的在线学习场景。

**Q：图中强调的关键性质“无偏”在数学上如何表达？**

**A：** 无偏性意味着随机抽取单样本计算出的梯度，其数学期望恰好等于所有样本参与计算的全量梯度。公式表达为：


$$E_\xi [\nabla_{\mathbf{w}} l(\mathbf{w}, \xi)] = \frac{1}{n} \sum_{i=1}^n \nabla_{\mathbf{w}} l(\mathbf{w}, \mathbf{x}_i) = \nabla f(\mathbf{w})$$

**Q：“无偏性”不就是样本期望等于真实期望吗？为什么它能解释 SGD 的合理性？**

**A：** 这一统计学概念在最优化理论中具有决定生死的含义，它是 SGD 能够成功收敛的理论基石，具体体现在以下三个核心层面：

* **消除系统性偏差，保证“大方向”绝对正确：** 单样本梯度充满噪声，每一次更新的方向可能偏左、偏右甚至短暂地使总损失变大。无偏性保证了这些随机噪声的**均值为零**。只要迭代步数足够多，正负误差会互相抵消，确保模型虽然在随机震荡，但总体轨迹必定不可阻挡地向真正的最优解收敛。如果梯度更新是有偏的，系统性误差会不断累积，最终必然导致模型发散或收敛到错误的解。
* **解释了引入“动态学习率”的绝对必要性：** 尽管期望是无偏的，但单样本梯度的噪声方差不随迭代衰减。如果保持恒定的学习率，参数到达最优解附近后会被这些不衰减的噪声不断弹开，永远只能震荡而无法真正收敛。因此，必须让学习率 $\eta_t$ 随时间衰减来抹平噪声方差的影响。常见的衰减形式包括多项式衰减（$\eta_t = \eta_0 \cdot t^{-\alpha}$）、指数衰减（$\eta_t = \eta_0 \cdot \beta^t$）以及分段常数。


* **提供收敛性证明的理论保障：** 依托无偏性和方差有界的假设，最优化理论可以严格证明 SGD 在期望意义上能实现目标函数值的下降。对于凸目标函数，SGD 的收敛率为 $O(1/\sqrt{t})$，对于强凸函数，收敛率为 $O(1/t)$。虽然单看收敛步数慢于全量梯度下降的线性收敛，但凭借单步极小的计算代价，SGD 成为高维大数据场景下性价比最高的优化算法。

## 11.5 小批量随机梯度下降

**动机：向量化与缓存**。深度学习框架（尤其 GPU）对"一大片整齐的矩阵乘法"优化极好；逐条处理单个样本既慢又浪费缓存。小批量（minibatch）是**计算效率**与**统计效率**的折中。

设批量 $B$ 的平均梯度 $\mathbf{g}_B = \frac{1}{|B|} \sum_{i \in B} \nabla l(\mathbf{w}, \mathbf{x}_i)$：

- **无偏**：$E[\mathbf{g}_B] = \nabla f(\mathbf{w})$；
- **方差随批量线性减小**：若单个样本梯度方差为 $\sigma^2$，则

$$
\mathrm{Var}\left[\mathbf{g}_B\right] = \sigma^2 / |B|
$$

但梯度计算代价也随 $|B|$ 线性增大，边际收益递减——所以批量不必太大（实践中常用 32–512）：

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \eta_t \mathbf{g}_B
$$

实现上就是训练循环里"按 batch 取数据 → 前向 → 反向 → `optimizer.step()`"，PyTorch 的 `DataLoader` 负责随机打乱与组 batch。

## 11.6 动量法

**问题：病态条件**。在 $f(\mathbf{w}) = 0.1 w_1^2 + 2 w_2^2$ 这类目标上，$w_2$ 方向曲率是 $w_1$ 的 20 倍：固定学习率要么在 $w_2$ 方向来回震荡，要么在 $w_1$ 方向龟速前进。

**方法**：把历史梯度做**指数加权平均（EMA）**，用它代替当前梯度：

$$
\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \mathbf{g}_t, \qquad \mathbf{w}_{t+1} = \mathbf{w}_t - \eta \mathbf{v}_t
$$

把递推展开可知 $\mathbf{v}_t$ 是过去所有梯度的加权和（权重 $\beta^k$ 递减，有效窗口约 $1/(1-\beta)$ 步：$\beta=0.5$ ≈ 平均过去 2 步，$\beta=0.9$ ≈ 平均过去 10 步）。效果：

- 相邻梯度**方向一致** → 互相增强 → 加速小曲率方向的"慢"；
- 方向**来回翻转** → 相互抵消 → 抑制大曲率方向的"震"。

（另有物理解释：$\mathbf{v}$ 是速度，负梯度是外力，$\beta$ 扮演摩擦系数——"小球在山坡上带惯性滚动"。）

PyTorch 中 `torch.optim.SGD(net.parameters(), lr=η, momentum=β)` 即为该实现。

**Q：动量法（Momentum）主要为了解决什么问题？**
**A：** 动量法主要为了解决优化过程中的“病态条件”问题。例如在目标函数 $f(\mathbf{w}) = 0.1w_1^2 + 2w_2^2$ 这类地形中，$w_2$ 方向的曲率是 $w_1$ 方向的 20 倍。如果在这种地形下使用固定的学习率，参数更新要么在曲率大的 $w_2$ 方向来回震荡，要么在曲率小的 $w_1$ 方向呈龟速前进。

**Q：动量法的核心机制和公式是什么？**
**A：** 动量法的核心思想是将历史梯度做**指数加权平均（EMA）**，并用它来代替当前步的原始梯度进行参数更新。其数学递推公式分为两步：

1. 更新动量变量：$\mathbf{v}_t = \beta \mathbf{v}_{t-1} + \mathbf{g}_t$

2. 更新参数：$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \mathbf{v}_t$


**Q：公式中指数加权平均（EMA）的超参数 $\beta$ 有什么具体数学含义？**
**A：** 将 $\mathbf{v}_t$ 的递推公式展开可知，它实际上是过去所有梯度的加权和，其中历史梯度的权重以 $\beta^k$ 的形式递减。$\beta$ 决定了有效记忆的历史步数，其有效窗口大约为 $1/(1-\beta)$ 步。

* 当 $\beta = 0.5$ 时，约等于平均了过去 2 步的梯度。


* 当 $\beta = 0.9$ 时，约等于平均了过去 10 步的梯度。



**Q：动量法如何巧妙化解病态条件下的“震荡”与“龟速”问题？**
**A：** 动量法通过历史梯度的累积，产生了两种截然不同的调节效果：

* **加速小曲率方向的“慢”：** 当相邻步的梯度方向一致时，它们会在动量变量中互相增强，从而加速模型在平坦方向上的更新进度。


* **抑制大曲率方向的“震”：** 当相邻步的梯度方向来回翻转时，正负梯度会在加权求和中相互抵消，从而有效抑制模型在该方向上的剧烈震荡。



**Q：动量法在物理学上该如何直观理解？**
**A：** 在物理模型中，整个寻优过程可以直观地理解为“小球在山坡上带惯性滚动”。其中，动量变量 $\mathbf{v}$ 代表速度，负梯度代表施加的外力，而参数 $\beta$ 则扮演了摩擦系数的角色。

**Q：在实际的代码框架（如 PyTorch）中如何实现动量法？**
**A：** 在 PyTorch 中，可以直接通过在标准 SGD 优化器中传入 `momentum` 参数来实现，代码即为 `torch.optim.SGD(net.parameters(), lr=\eta, momentum=\beta)`。

## 11.7 AdaGrad 算法

**问题：稀疏特征**。比如词频特征中罕见词出现极少，其参数梯度长期偏小 → 应该给它们**更大**的学习率；高频特征则相反。需要**按坐标自适应学习率**。

**算法**：把历史梯度平方**逐坐标累加**到 $\mathbf{s}$，用它缩放有效学习率：

$$
\mathbf{s}_t = \mathbf{s}_{t-1} + \mathbf{g}_t^2, \qquad
\mathbf{g}_t' = \frac{\eta}{\sqrt{\mathbf{s}_t} + \epsilon} \odot \mathbf{g}_t, \qquad
\mathbf{w}_{t+1} = \mathbf{w}_t - \mathbf{g}_t'
$$

（$\odot$ 为逐元素乘，$\mathbf{g}^2$ 为逐元素平方。）

- 梯度持续很大的坐标：$\mathbf{s}$ 大 → 有效学习率小 → 抑制震荡；
- 梯度长期偏小的坐标（稀疏特征）：$\mathbf{s}$ 小 → 有效学习率大 → 快速"补课"。

**缺陷**：$\mathbf{s}$ 单调累加、永不减小，有效学习率随时间**必然衰减到 0**——训练后期几乎不再更新。这个缺陷由 RMSProp 修复。PyTorch：`torch.optim.Adagrad`。

## 11.8 RMSProp 算法

**改动只有一处**：把 AdaGrad 的"累加"换成**指数加权移动平均（EMA）**，让 $\mathbf{s}$ 具备"遗忘"能力：

$$
\mathbf{s}_t = \gamma \mathbf{s}_{t-1} + (1-\gamma) \mathbf{g}_t^2, \qquad
\mathbf{w}_{t+1} = \mathbf{w}_t - \frac{\eta}{\sqrt{\mathbf{s}_t} + \epsilon} \odot \mathbf{g}_t
$$

$\mathbf{s}$ 只反映最近约 $1/(1-\gamma)$ 步的梯度量级，学习率**不会一路衰减到零**，训练中后期依然保持有效步长。$\gamma$ 通常取 0.9。

PyTorch：`torch.optim.RMSprop(net.parameters(), lr=η, alpha=0.99)`（`alpha` 即上式中的 $\gamma$）。

> RMSProp 是 Hinton 在 Coursera 课程中随口提出的"经验方法"，却成了现代自适应优化器的基石。

## 11.9 Adadelta

Adadelta 是 AdaGrad 的另一种延伸，核心是两件事：

1. 与 RMSProp 一样，用 EMA 替代累加：$\mathbf{s}_t = \rho\, \mathbf{s}_{t-1} + (1-\rho)\, \mathbf{g}_t^2$；
2. **不使用学习率量纲**：更新量的单位应与参数一致，因此再用一个 EMA 记录"更新量平方" $\Delta\mathbf{x}$，把更新写为

$$
\mathbf{x}_t' = \frac{\sqrt{\Delta\mathbf{x}_{t-1} + \epsilon}}{\sqrt{\mathbf{s}_t + \epsilon}} \odot \mathbf{g}_t, \qquad
\Delta\mathbf{x}_t = \rho\, \Delta\mathbf{x}_{t-1} + (1-\rho)\, \left(\mathbf{x}_t'\right)^2
$$

由于 $\Delta\mathbf{x}$ 初值为 0，前若干步的等效学习率较大，Adadelta **不需要设置默认学习率**这一超参数（PyTorch 仍保留 `lr=1.0` 兜底）。PyTorch：`torch.optim.Adadelta`。

## 11.10 Adam 算法

Adam（Adaptive Moment Estimation）把前面所有技巧组合起来，是实践中最常见的默认优化器：

| 组合成分 | 来源 | 作用 |
| --- | --- | --- |
| 一阶矩 EMA $\mathbf{v}$ | 动量法（11.6） | 平滑梯度方向 |
| 二阶矩 EMA $\mathbf{s}$ | RMSProp（11.8） | 逐坐标缩放步长 |
| 偏差修正 | Adam 原创 | 修正 EMA 初期偏向 0 的估计 |

$$
\mathbf{v}_t = \beta_1 \mathbf{v}_{t-1} + (1-\beta_1) \mathbf{g}_t, \qquad
\mathbf{s}_t = \beta_2 \mathbf{s}_{t-1} + (1-\beta_2) \mathbf{g}_t^2
$$

因为 $\mathbf{v}_0 = \mathbf{s}_0 = \mathbf{0}$，训练初期 EMA 估计偏小，用 $1-\beta^t$ 因子做**无偏化（偏差修正）**：

$$
\hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1-\beta_1^t}, \qquad
\hat{\mathbf{s}}_t = \frac{\mathbf{s}_t}{1-\beta_2^t}, \qquad
\mathbf{w}_{t+1} = \mathbf{w}_t - \frac{\eta\, \hat{\mathbf{v}}_t}{\sqrt{\hat{\mathbf{s}}_t} + \epsilon}
$$

默认超参数：$\beta_1 = 0.9$、$\beta_2 = 0.999$、$\epsilon = 10^{-6}$。

变体 **Yogi**：把二阶矩更新改为

$$
\mathbf{s}_t = \mathbf{s}_{t-1} + (1-\beta_2)\left(\mathbf{g}_t^2 - \mathbf{s}_{t-1}\right) \odot \mathrm{sign}\left(\mathbf{g}_t^2 - \mathbf{s}_{t-1}\right)
$$

用 sign 控制步长，防止大梯度使 $\mathbf{s}$ 增长过快（爆炸）。

PyTorch：`torch.optim.Adam(net.parameters(), lr=η, betas=(0.9, 0.999))`。

## 11.11 学习率调度器

**一个简单的问题**：固定学习率的算法在训练后期会在最优点附近来回震荡，无法精细收敛——即使其它一切都对，**学习率本身也需要"随时间调度"**。

常用策略（PyTorch 均在 `torch.optim.lr_scheduler` 中）：

| 策略 | 公式 / 做法 | 适用场景 |
| --- | --- | --- |
| 阶梯衰减 `StepLR` / `MultiStepLR` | 每 $\gamma$ 个 epoch 把 lr 乘 0.1 | 最常用基线 |
| 指数衰减 `ExponentialLR` | $\eta_t = \eta_0 \gamma^t$ | 平滑衰减 |
| 余弦退火 `CosineAnnealingLR` | $\eta_t = \eta_{\min} \frac{1}{2}\left(1 + \cos\left(\frac{\pi t}{T}\right)\right)$ | 现代 CV 训练标配 |
| 暖重启 `CosineAnnealingWarmRestarts` | 周期性把 lr 拉回初始值（SGDR） | 跳出次优、模型集成 |
| OneCycleLR | 先升后降（warmup + anneal） | 快速收敛、大 batch 训练 |
| 预热 warmup | 起始几步 lr 从 0 线性升到目标 | Transformer 必备 |

经验法则：**前几步 warmup 防止初期大梯度爆炸 → 中期较大 lr 快速前进 → 后期小 lr 精细收敛**。调度器与优化器解耦：每个 epoch / step 调用 `scheduler.step()` 即可。

## 全章小结：一条演化线

**GD →（去计算瓶颈）SGD →（平滑震荡）动量法 →（逐维自适应）AdaGrad →（修衰减）RMSProp →（去 lr 量纲）AdaDelta →（集大成）Adam →（叠加调度）LR Scheduler**

每个优化器都只解决上一代的一个痛点：

| 痛点 | 药方 |
| --- | --- |
| 全量梯度太贵 | SGD / 小批量 SGD（无偏 + 方差 $\propto 1/b$） |
| 大曲率方向震荡、小曲率方向慢 | 动量法（历史梯度 EMA） |
| 稀疏特征学得慢 | AdaGrad（逐维自适应 lr） |
| AdaGrad 学习率衰减到 0 | RMSProp（EMA 有"遗忘"） |
| 更新量与参数量纲不一致 | AdaDelta（单位校正，免默认 lr） |
| 初期 EMA 估计有偏 | Adam（偏差修正 = 动量 + RMSProp） |
| 固定 lr 后期震荡 | 学习率调度器（衰减 / 余弦 / warmup） |

实践建议：**首选 Adam（或 AdamW）+ 余弦退火 / 阶梯衰减起步**；追求极致泛化与吞吐时对比 SGD + momentum（CV 常见）；Transformer 类模型务必 warmup。

## 动手验证

### 实验 1：病态条件问题上的 GD vs 动量法

目标函数 $f(\mathbf{w}) = 0.1 w_1^2 + 2 w_2^2$（两个方向的曲率相差 20 倍）。固定学习率 $\eta = 0.4$：普通梯度下降在 $w_2$ 方向来回震荡、$w_1$ 方向进展缓慢；加入动量 $\beta = 0.5$ 后，震荡被历史梯度相互抵消，轨迹平滑、更快地接近原点。

In [ ]:
import torch
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  # Windows 中文字体
plt.rcParams['axes.unicode_minus'] = False

def f(w):          # 病态条件目标函数
    return 0.1 * w[0] ** 2 + 2 * w[1] ** 2

def grad(w):       # 解析梯度
    return torch.tensor([0.2 * w[0], 4.0 * w[1]])

def train_2d(eta, beta=None, steps=20):
    """beta=None 为普通梯度下降；否则为动量法"""
    w = torch.tensor([-2.0, -2.0])
    v = torch.zeros(2)
    path = [w.clone()]
    for _ in range(steps):
        g = grad(w)
        if beta is not None:
            v = beta * v + g   # 动量 = 历史梯度 EMA
            w = w - eta * v
        else:
            w = w - eta * g
        path.append(w.clone())
    return torch.stack(path)

gd = train_2d(0.4)               # 普通梯度下降
mom = train_2d(0.4, beta=0.5)    # 动量法

# 画等高线 + 两条轨迹
x = torch.linspace(-6, 6, 200)
y = torch.linspace(-6, 6, 200)
XX, YY = torch.meshgrid(x, y, indexing='xy')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, path, title in zip(axes, [gd, mom],
                           ['梯度下降  η=0.4', '动量法  η=0.4, β=0.5']):
    ax.contour(XX.numpy(), YY.numpy(), (0.1 * XX ** 2 + 2 * YY ** 2).numpy(),
               20, cmap='viridis', alpha=0.6)
    ax.plot(path[:, 0], path[:, 1], 'o-', color='crimson', ms=4)
    ax.plot(path[0, 0], path[0, 1], 's', color='blue', ms=8, label='起点')
    ax.plot(path[-1, 0], path[-1, 1], '*', color='gold', ms=14, label='终点')
    ax.set_title(title)
    ax.legend()
    ax.set_xlabel('w1')
    ax.set_ylabel('w2')
plt.tight_layout()
plt.show()

print('GD 最终位置   :', gd[-1].tolist())
print('动量法最终位置:', mom[-1].tolist(), '（更接近最优 [0, 0]）')

### 实验 2：五种优化器收敛速度对比

用同一个带噪声的线性回归任务（$y = \mathbf{X}\mathbf{w} + b + \epsilon$），分别用 SGD、动量法、AdaGrad、RMSProp、Adam 训练**结构相同、初始化相同**的单层网络，比较损失下降速度（对数坐标下数量级差异一目了然）。

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

# 带噪声的线性回归数据：y = Xw + b + ε
n, d = 1000, 2
X = torch.randn(n, d)
w_true, b_true = torch.tensor([[2.0], [-3.0]]), 4.2
y = X @ w_true + b_true + 0.01 * torch.randn(n, 1)

def make_opt(name, net):
    """每种优化器使用相同的初始学习率 0.05"""
    lr = 0.05
    return {
        'SGD':      lambda: torch.optim.SGD(net.parameters(), lr=lr),
        'Momentum': lambda: torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9),
        'AdaGrad':  lambda: torch.optim.Adagrad(net.parameters(), lr=lr),
        'RMSProp':  lambda: torch.optim.RMSprop(net.parameters(), lr=lr),
        'Adam':     lambda: torch.optim.Adam(net.parameters(), lr=lr),
    }[name]()

loss_fn = nn.MSELoss()
histories = {}
for name in ['SGD', 'Momentum', 'AdaGrad', 'RMSProp', 'Adam']:
    net = nn.Linear(d, 1)              # 每个优化器都重新初始化同结构网络
    opt = make_opt(name, net)
    losses = []
    for epoch in range(50):            # 全批量训练，突出优化器本身的差异
        opt.zero_grad()
        loss = loss_fn(net(X), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    histories[name] = losses

plt.figure(figsize=(8, 4.5))
for name, losses in histories.items():
    plt.semilogy(losses, label=name)   # 对数坐标更能看出数量级差异
plt.xlabel('epoch')
plt.ylabel('MSE (log)')
plt.title('不同优化器的收敛速度对比')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print('初始损失:', {k: round(v[0], 2) for k, v in histories.items()})
print('最终损失:', {k: round(v[-1], 4) for k, v in histories.items()})

> **接下来**：按路线图逐节深入——建议从 [11.1 优化和深度学习](https://zh-v2.d2l.ai/chapter_optimization/optimization-intro.html) 开始，弄清"为什么非凸优化这么难"，再顺着梯度下降 → SGD → 小批量 → 各优化器一路学下去。